In [ ]:
##########################################################################################
#import data
##########################################################################################

#import libraries
import pandas as pd

#import the dataset
df = pd.read_csv('medical_insurance.csv', na_values = "?")
print("Raw shape:", df.shape)   #prints the number of rows and columns in the dataset
print(df.head())                #prints the first 5 rows of the dataset
df_original = df.copy(deep=True)#makes a fully independent copy 

In [ ]:
##########################################################################################
#Check for missing values and duplicates
##########################################################################################

#check for missing values
missing = df_original.isna().sum()
print(missing[missing > 0].sort_values(ascending=False))

#check for duplicate rows
print("Duplicate rows:", df_original.duplicated().sum())

In [ ]:
##########################################################################################
#Transformation: Converting categorical variables to numerical values for analysis
##########################################################################################

#transforming the data by converting str variables to int64
def clean_data(data):
    clean = data.copy(deep=True)
    clean["sex"] = clean["sex"].replace({"M": 0, "F": 1, "m": 0, "f": 1})  # Convert sex to numerical values
    clean["smoker"] = clean["smoker"].replace({"Never": 0, "Former": 1, "Current": 2})  #convert smoker to numerical values
    clean["alcohol_freq"] = clean["alcohol_freq"].replace({"None": 0, "Occasionally": 1, "Weekly": 2, "Daily": 3})  # Convert alcohol frequency to numerical values
    return clean

In [ ]:
# ENGE707 Project Phase-I: Task 3 - Data Cleansing and Transformation
# Dataset: medical_insurance.csv
# Target: risk_score (continuous, 0-1)

import pandas as pd
import numpy as np


df = pd.read_csv("medical_insurance.csv", keep_default_na=False, na_values=[""])
 
print(f"Loaded {df.shape[0]:,} rows x {df.shape[1]} columns")

In [ ]:
# To detect duplicates
assert df["person_id"].is_unique, "Duplicate person_id found"
assert df.duplicated().sum() == 0, "Duplicate rows found"

# alcohol_freq had problems with "none" being a missing value
# Confirm that no missing values remain
missing = df.isna().sum()
print("\nColumns with missing values after fixing the None/NaN parsing issue:")
print(missing[missing > 0] if missing.sum() else "(none)")

In [ ]:
# Checking for if chronic_count should equal the sum of the individual disease flags

disease_cols = [
    "hypertension", "diabetes", "asthma", "copd", "cardiovascular_disease",
    "cancer_history", "kidney_disease", "liver_disease", "arthritis",
    "mental_health",
]
mismatch = (df[disease_cols].sum(axis=1) != df["chronic_count"]).sum()
print(f"\nRows where chronic_count disagrees with disease-flag sum: {mismatch}")

In [ ]:
# A small nunber of records show age 0-1. Keeping them because they can be listed as dependants on a policy, but flagging them to keep decision visible.

df["age_flag_infant"] = df["age"] <= 1
print(f"\nRecords flagged as infants (age<=1), kept but flagged: {df['age_flag_infant'].sum()}")

In [ ]:
# Some BMI values are at 12.0, below any possible adult BMI, may be floor/clipping artifact from date generation, flagging for now.

BMI_LOW, BMI_HIGH = 13, 55  # plausible physiological range
df["bmi_flag_outlier"] = ~df["bmi"].between(BMI_LOW, BMI_HIGH)
print(f"BMI values outside [{BMI_LOW}, {BMI_HIGH}] flagged: {df['bmi_flag_outlier'].sum()}")

In [ ]:
# Excluding values that are a direct result of risk_score, since they are unnecessary for predicting the target value

LEAKAGE_COLS = [
    "is_high_risk", "annual_premium", "monthly_premium", "annual_medical_cost",
    "claims_count", "avg_claim_amount", "total_claims_paid",
]
print(f"\nColumns flagged as target-leakage risk (exclude from explanatory model): {LEAKAGE_COLS}")

In [ ]:
categorical_cols = [
    "sex", "region", "urban_rural", "education", "marital_status",
    "employment_status", "smoker", "alcohol_freq", "plan_type", "network_tier",
]
for col in categorical_cols:
    df[col] = df[col].str.strip()  # guard against stray whitespace

# Ordinal columns will be encoded since they have a meaningful order. To be done tranformation.

In [ ]:
# Save Cleaned dataset

df.to_csv("medical_insurance_cleaned.csv", index=False)
print(f"\nSaved cleaned dataset: medical_insurance_cleaned.csv ({df.shape[0]:,} rows x {df.shape[1]} columns)")

In [ ]:
#Task 4 - Data Visualization and Analysis - all analysis is done using the clean dataset
import pandas as pd
import matplotlib.pyplot as plt
df = pd.read_csv("medical_insurance_cleaned.csv")
#Key categories to analyze: risk score vs controllable variables
#such as bmi, income, smoker, alcohol_freq, urban_rural and how they affect risk score
# by understanding the relationships, we may be able to identify areas for intervention to reduce risk score and improve patient outcomes.

In [ ]:
#Begin with summaries of the data to get a sense of the distribution of the variables
num_features = ["bmi", "income", "risk_score"]
categorical_features = ["smoker", "alcohol_freq", "urban_rural"]
print("Summary statistics for the numerical variables in the dataset:")
print(df[num_features].describe())



In [ ]:
#the majority of patients have a BMI between 20 and 40, with a few outliers having a BMI above 40. The median BMI is around 27, which is considered slightly overweight.
skew_results = df[num_features].skew().round(2)
print(skew_results)

#then we can visualize the relationships between the risk score and the controllable variables using scatter plots for numerical variables and box plots for categorical variables. This will help us identify any patterns or trends in the data that may be useful for intervention strategies.

In [ ]:
#Histograms of the numerical variables to visualize the distribution of the data
import matplotlib.pyplot as plt

plt.hist(df["bmi"].dropna(), bins=50)
plt.xlabel("BMI"); plt.ylabel("Number of patients")
plt.title("Distribution of BMI")
plt.show()

plt.hist(df["income"].dropna(), bins=100)
plt.xlabel("Income"); plt.ylabel("Number of patients")
plt.title("Distribution of Income")
plt.show()

plt.hist(df["risk_score"].dropna(), bins=45)
plt.xlabel("Risk Score"); plt.ylabel("Number of patients")
plt.title("Distribution of Risk Score")
plt.show()

#from the histograms, we can see that age is roughly normally distributed, with a slight skew to the right. BMI is also roughly normally distributed, with a slight skew to the right. Risk score is heavily skewed to the right, with most patients having a low risk score and a few outliers having a high risk score. Chronic count is also heavily skewed to the right, with most patients having a low number of chronic conditions and a few outliers having a high number of chronic conditions.

In [ ]:
#How many patients have a risk score of 0, and how many have a risk score of 1 or higher
print("people with risk score = 0: " + str(df[df["risk_score"] == 0].shape[0]))
print("people with risk score >= 1: " + str(df[df["risk_score"] >= 1].shape[0]))

#1131 patients have a risk score of 0, and 5241 patients have a risk score of 1 or higher.

In [ ]:
import numpy as np

def calculate_kruskal(df, group_col, target_col='risk_score'):
    # Drop rows with missing values
    clean = df[[group_col, target_col]].dropna()
    
    # Extract unique categories
    categories = clean[group_col].unique()
    k = len(categories)
    N = len(clean)
    
    # Combine and rank all target values (assign average ranks for ties)
    vals = clean[target_col].values
    ranks = np.argsort(np.argsort(vals)) + 1.0
    
    # Calculate sum of ranks squared divided by sample size for each group
    rank_sum_sq = 0
    for cat in categories:
        mask = (clean[group_col] == cat).values
        n_i = np.sum(mask)
        R_i = np.sum(ranks[mask])
        rank_sum_sq += (R_i ** 2) / n_i
        
    # Calculate Kruskal-Wallis H statistic
    H = (12.0 / (N * (N + 1))) * rank_sum_sq - 3.0 * (N + 1)
    
    # Degrees of freedom = number of groups - 1
    df_val = k - 1
    
    print(f"Feature: {group_col:<15} | H-Stat: {H:8.4f} | df: {df_val}")
    return H

# Run test on all categorical features
for col in ['urban_rural', 'smoker', 'alcohol_freq']:
    calculate_kruskal(df, col)

In [ ]:
import numpy as np
 
def calculate_anova(df, group_col, target_col='risk_score'):
    clean = df[[group_col, target_col]].dropna()
    categories = clean[group_col].unique()
    
    overall_mean = np.mean(clean[target_col].values)
    k = len(categories)
    N = len(clean)
    
    ss_between = 0
    ss_within = 0
    
    for cat in categories:
        group_vals = clean[clean[group_col] == cat][target_col].values
        n_i = len(group_vals)
        group_mean = np.mean(group_vals)
        
        # Variance between group mean and overall mean
        ss_between += n_i * ((group_mean - overall_mean) ** 2)
        # Variance inside the group
        ss_within += np.sum((group_vals - group_mean) ** 2)
        
    ms_between = ss_between / (k - 1)
    ms_within = ss_within / (N - k)
    
    # Calculate F statistic
    F = ms_between / ms_within
    print(f"Feature: {group_col:<15} | F-Stat: {F:8.4f} | df: ({k-1}, {N-k})")
    return F

for col in ['urban_rural', 'smoker', 'alcohol_freq']:
    calculate_anova(df, col)

In [ ]:
#BOXPLOT OF SMOKER

import matplotlib.pyplot as plt

# Group data by smoker category
groups = df.groupby('smoker', observed=False)['risk_score']
labels = [str(name) for name, _ in groups]
data = [group.dropna().values for _, group in groups]

# Calculate exact medians for annotation
medians = [np.median(d) for d in data]

fig, ax = plt.subplots(figsize=(8, 5))

# Draw Boxplot (The center line inside each box IS the median)
bp = ax.boxplot(
    data, 
    tick_labels=labels, 
    patch_artist=True,
    showmeans=True, # White circle = Mean, Black line = Median
    meanprops={"marker": "o", "markerfacecolor": "white", "markeredgecolor": "black"},
    medianprops={"color": "red", "linewidth": 2.5}, # Highlight median in bold red
    boxprops={"facecolor": "#2b5c8f", "alpha": 0.6}
)

# Annotate exact median values directly on top of the median lines
for i, median_val in enumerate(medians):
    ax.text(
        i + 1, median_val + 0.02, f"Median: {median_val:.3f}", 
        ha='center', va='bottom', fontweight='bold', color='darkred'
    )

ax.set_title('Risk Score Median Comparison by Smoking Status', fontweight='bold')
ax.set_xlabel('Smoking Status', fontweight='bold')
ax.set_ylabel('Risk Score', fontweight='bold')
ax.grid(True, axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()